In [ ]:
import numpy as np
import geopandas as gpd
import folium

In [ ]:
# Cargar los archivos GeoJSON
import os

data_path = 'data/'
geojson_files = {
    'farmacias': 'farmacia.geojson',
    'hospitales': 'hospital.geojson',
    'poblaciones': 'poblaciones.geojson',
    'salud': 'resultados_salud.geojson',
    'zonas_verdes': 'zona_verde.geojson'
}

gdfs = {} 

for name, file in geojson_files.items():
    path = os.path.join(data_path, file)
    if os.path.exists(path):
        # Cargamos el archivo y forzamos una copia para evitar problemas de referencia
        nuevo_gdf = gpd.read_file(path).copy()
        
        # Aseguramos el CRS 4326 inmediatamente
        if nuevo_gdf.crs != "EPSG:4326":
            nuevo_gdf = nuevo_gdf.to_crs(epsg=4326)
            
        gdfs[name] = nuevo_gdf
        print(f"Cargado {name}: {len(gdfs[name])} filas únicas.")

Cargado farmacias: 9279 filas únicas.
Cargado hospitales: 9279 filas únicas.
Cargado poblaciones: 9279 filas únicas.
Cargado salud: 9279 filas únicas.
Cargado zonas_verdes: 9279 filas únicas.


In [ ]:
# Calcular el centro del mapa basado en todos los datos
all_bounds = []
for gdf in gdfs.values():
    if not gdf.empty:
        bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
        all_bounds.append(bounds)

if all_bounds:
    all_bounds = np.array(all_bounds)
    minx, miny = all_bounds[:, :2].min(axis=0)
    maxx, maxy = all_bounds[:, 2:].max(axis=0)
    center_lon = (minx + maxx) / 2
    center_lat = (miny + maxy) / 2
    print(f"Centro del mapa: {center_lat:.4f}, {center_lon:.4f}")
else:
    center_lat, center_lon = 37.5, -4.5  # Centro aproximado de Andalucía
    print("Usando centro por defecto")

Centro del mapa: 37.3035, -4.5204


In [ ]:
mapa = folium.Map(location=[center_lat, center_lon],zoom_start=7)
# 1. Definir colores para cada categoría
colores = {
    'farmacias': 'red',
    'hospitales': 'blue',
    'poblaciones': 'cadetblue',
    'salud': 'orange',
    'zonas_verdes': 'green'
}
# 2. Iterar sobre los GeoDataFrames cargados
for name, gdf in gdfs.items():
    # Creamos un grupo ÚNICO para este archivo GeoJSON actual
    # Al estar dentro del bucle 'for name', este 'group' se vacía en cada vuelta
    group = folium.FeatureGroup(name=f"Capa: {name.capitalize()}", show=True)
    
    # Obtenemos el color específico para este grupo
    color_capa = colores.get(name, 'gray')

    # 3. AÑADIR SOLO LOS PUNTOS DE 'GDF' (el actual del bucle) AL 'GROUP'
    for _, row in gdf.iterrows():
        try:
            # Extraer coordenadas según el tipo de geometría
            if row.geometry.geom_type == 'Point':
                location = [row.geometry.y, row.geometry.x]
            else:
                centro = row.geometry.centroid
                location = [centro.y, centro.x]

            # EL PASO CLAVE: .add_to(group)
            # Asegúrate de NO usar .add_to(mapa) aquí dentro
            folium.CircleMarker(
                location=location,
                radius=5,
                popup=f"Capa: {name}<br>Nombre: {row.get('nombre', 'Sin nombre')}",
                color=color_capa,
                fill=True,
                fill_color=color_capa,
                fill_opacity=0.7
            ).add_to(group) # <--- SE AÑADE AL GRUPO, NO AL MAPA DIRECTAMENTE
            
        except Exception:
            continue

    # 4. AÑADIR EL GRUPO (YA LLENO) AL MAPA
    # Esto debe ocurrir una sola vez por cada archivo GeoJSON
    group.add_to(mapa)

# 5. CONTROL DE CAPAS AL FINAL (FUERA DE TODOS LOS BUCLES)
folium.LayerControl(collapsed=False).add_to(mapa)

# 6. Guardar y visualizar
mapa.save('map/mapa_andalucia.html')